In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
data = np.load("../models/preprocessed_data.npz")

X_train_early = data["X_train_early"]
X_val_early = data["X_val_early"]
X_test_early = data["X_test_early"]

X_train_late = data["X_train_late"]
X_val_late = data["X_val_late"]
X_test_late = data["X_test_late"]

y_train = data["y_train"]
y_val = data["y_val"]
y_test = data["y_test"]

print("Data loaded successfully!")

print("Early test shape:", X_test_early.shape)
print("Late test shape:", X_test_late.shape)
print("Test target shape:", y_test.shape)

Data loaded successfully!
Early test shape: (98, 56)
Late test shape: (98, 58)
Test target shape: (98,)


In [3]:
print("Test samples:", len(y_test))
print("Early features:", X_test_early.shape[1])
print("Late features:", X_test_late.shape[1])

Test samples: 98
Early features: 56
Late features: 58


In [4]:
# Early prediction model
lr_early = LogisticRegression(max_iter=1000)
lr_early.fit(X_train_early, y_train)

y_pred_lr_early = lr_early.predict(X_test_early)


# Late prediction model
lr_late = LogisticRegression(max_iter=1000)
lr_late.fit(X_train_late, y_train)

y_pred_lr_late = lr_late.predict(X_test_late)

In [5]:
print("EARLY PREDICTION - LOGISTIC REGRESSION")

print("Accuracy:", accuracy_score(y_test, y_pred_lr_early))
print("Macro F1:", f1_score(y_test, y_pred_lr_early, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr_early))


print("\nLATE PREDICTION - LOGISTIC REGRESSION")

print("Accuracy:", accuracy_score(y_test, y_pred_lr_late))
print("Macro F1:", f1_score(y_test, y_pred_lr_late, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr_late))

EARLY PREDICTION - LOGISTIC REGRESSION
Accuracy: 0.5306122448979592
Macro F1: 0.3596165739022882

Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.70      0.67        54
           1       0.00      0.00      0.00         7
           2       0.37      0.45      0.41        22
           3       0.57      0.27      0.36        15

    accuracy                           0.53        98
   macro avg       0.39      0.36      0.36        98
weighted avg       0.52      0.53      0.51        98


LATE PREDICTION - LOGISTIC REGRESSION
Accuracy: 0.7653061224489796
Macro F1: 0.6790293040293041

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.94      0.86        54
           1       0.60      0.43      0.50         7
           2       0.59      0.45      0.51        22
           3       1.00      0.73      0.85        15

    accuracy                           0.77        

In [6]:
def build_dnn(input_features):

    model = Sequential([
        Dense(64, activation='relu', input_shape=(input_features,)),
        Dropout(0.3),

        Dense(32, activation='relu'),
        Dropout(0.2),

        Dense(4, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [7]:
early_dnn = build_dnn(X_train_early.shape[1])

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

early_dnn.fit(
    X_train_early,
    y_train,
    validation_data=(X_val_early, y_val),
    epochs=100,
    batch_size=16,
    callbacks=[early_stopping],
    verbose=1
)

y_pred_dnn_early = np.argmax(
    early_dnn.predict(X_test_early),
    axis=1
)

Epoch 1/100


c:\Users\Parv\Desktop\DNN Project\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.3612 - loss: 1.3775 - val_accuracy: 0.5464 - val_loss: 1.1292
Epoch 2/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5110 - loss: 1.2169 - val_accuracy: 0.5464 - val_loss: 1.0940
Epoch 3/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5066 - loss: 1.1460 - val_accuracy: 0.5361 - val_loss: 1.0740
Epoch 4/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5198 - loss: 1.1063 - val_accuracy: 0.5361 - val_loss: 1.0552
Epoch 5/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5595 - loss: 1.0403 - val_accuracy: 0.5464 - val_loss: 1.0385
Epoch 6/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5463 - loss: 1.0732 - val_accuracy: 0.5464 - val_loss: 1.0202
Epoch 7/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5837 - loss: 1.0111 - val_accuracy: 0.5361 - val_loss: 1.0065
Epoch 8/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5595 - loss: 1.0329 - val_accuracy: 0.5567 - val_loss: 1.0

In [8]:
late_dnn = build_dnn(X_train_late.shape[1])

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

late_dnn.fit(
    X_train_late,
    y_train,
    validation_data=(X_val_late, y_val),
    epochs=100,
    batch_size=16,
    callbacks=[early_stopping],
    verbose=1
)

y_pred_dnn_late = np.argmax(
    late_dnn.predict(X_test_late),
    axis=1
)

Epoch 1/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.4053 - loss: 1.3500 - val_accuracy: 0.5361 - val_loss: 1.1121
Epoch 2/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5441 - loss: 1.1302 - val_accuracy: 0.5670 - val_loss: 1.0192
Epoch 3/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5529 - loss: 1.0491 - val_accuracy: 0.6082 - val_loss: 0.9201
Epoch 4/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5947 - loss: 0.9883 - val_accuracy: 0.6495 - val_loss: 0.8395
Epoch 5/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6278 - loss: 0.8983 - val_accuracy: 0.6495 - val_loss: 0.7650
Epoch 6/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6322 - loss: 0.8558 - val_accuracy: 0.7010 - val_loss: 0.7185
Epoch 7/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6498 - loss: 0.7826 - val_accuracy: 0.7010 - val_loss: 0.6812
Epoch 8/100
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6894 - loss: 0.7384 - val_accuracy: 0.7113 - v

In [13]:
# Save early prediction model
early_dnn.save("../models/dnn_early.keras")

# Save late prediction model
late_dnn.save("../models/dnn_late.keras")

print("Both DNN models saved successfully!")

Both DNN models saved successfully!


In [9]:
print("EARLY PREDICTION - DNN")

print("Accuracy:", accuracy_score(y_test, y_pred_dnn_early))
print("Macro F1:", f1_score(y_test, y_pred_dnn_early, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dnn_early))


print("\nLATE PREDICTION - DNN")

print("Accuracy:", accuracy_score(y_test, y_pred_dnn_late))
print("Macro F1:", f1_score(y_test, y_pred_dnn_late, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dnn_late))

EARLY PREDICTION - DNN
Accuracy: 0.5204081632653061
Macro F1: 0.2976715983507155

Classification Report:
              precision    recall  f1-score   support

           0       0.56      0.81      0.66        54
           1       0.00      0.00      0.00         7
           2       0.22      0.09      0.13        22
           3       0.50      0.33      0.40        15

    accuracy                           0.52        98
   macro avg       0.32      0.31      0.30        98
weighted avg       0.43      0.52      0.45        98


LATE PREDICTION - DNN
Accuracy: 0.7244897959183674
Macro F1: 0.6622832722832723

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.87      0.80        54
           1       0.80      0.57      0.67         7
           2       0.65      0.59      0.62        22
           3       0.70      0.47      0.56        15

    accuracy                           0.72        98
   macro avg       0.72      

c:\Users\Parv\Desktop\DNN Project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Parv\Desktop\DNN Project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Parv\Desktop\DNN Project\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

In [10]:
comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Scenario": "Early",
        "Accuracy": accuracy_score(y_test, y_pred_lr_early),
        "Macro F1": f1_score(y_test, y_pred_lr_early, average="macro")
    },
    {
        "Model": "Logistic Regression",
        "Scenario": "Late",
        "Accuracy": accuracy_score(y_test, y_pred_lr_late),
        "Macro F1": f1_score(y_test, y_pred_lr_late, average="macro")
    },
    {
        "Model": "DNN",
        "Scenario": "Early",
        "Accuracy": accuracy_score(y_test, y_pred_dnn_early),
        "Macro F1": f1_score(y_test, y_pred_dnn_early, average="macro")
    },
    {
        "Model": "DNN",
        "Scenario": "Late",
        "Accuracy": accuracy_score(y_test, y_pred_dnn_late),
        "Macro F1": f1_score(y_test, y_pred_dnn_late, average="macro")
    }
])

comparison

,Model,Scenario,Accuracy,Macro F1
0,Logistic Regression,Early,0.530612,0.359617
1,Logistic Regression,Late,0.765306,0.679029
2,DNN,Early,0.520408,0.297672
3,DNN,Late,0.724490,0.662283


In [11]:
best_model = comparison.loc[comparison["Macro F1"].idxmax()]

print("Best Model:")
print(best_model)

Best Model:
Model       Logistic Regression
Scenario                   Late
Accuracy               0.765306
Macro F1               0.679029
Name: 1, dtype: object


In [12]:
comparison.to_csv("../models/final_model_comparison.csv", index=False)

print("Final comparison saved successfully!")

Final comparison saved successfully!


## Final Evaluation and Observations

All four models were evaluated on the same independent test
dataset containing 98 student records.

The early prediction scenario excludes G1 and G2, while the
late prediction scenario includes these previous academic grades.

The results show that late prediction achieved better performance
than early prediction for both Logistic Regression and DNN.

Logistic Regression achieved the highest test accuracy of 76.53%
and Macro F1 score of 0.6790 in the late prediction scenario.

The results also demonstrate that a simpler traditional machine
learning model can outperform a Deep Neural Network on a
relatively small dataset.

The findings highlight the importance of previous academic
performance and the need to evaluate models using multiple metrics.

In [14]:
import os

files = [
    "../models/dnn_early.keras",
    "../models/dnn_late.keras",
    "../models/preprocessor_early.pkl",
    "../models/preprocessor_late.pkl",
    "../models/label_encoder.pkl"
]

for file in files:
    print(file, "->", os.path.exists(file))

../models/dnn_early.keras -> True
../models/dnn_late.keras -> True
../models/preprocessor_early.pkl -> True
../models/preprocessor_late.pkl -> True
../models/label_encoder.pkl -> True
